# Evaluating Arrays

Arrays need an **alignment** step before scoring: which gold element pairs with which
extracted element? The evaluator supports three strategies:

1. **Ordered** (default) -- Position matters. `"x-eval-align": {"ordered": true}`
2. **Key-field** -- match by a unique identifier field. Order doesn't matter.  `"x-eval-align": {"match_by": "key_field", "key": "name"}`
3. **Hungarian** -- optimal bipartite matching. Order doesn't matter, no key field needed, optimizes for best F1.     `"x-eval-align": {"match_by": "hungarian"}`

After alignment, each matched pair is scored recursively using the `items` schema.
Unmatched gold elements are **omissions**. Unmatched extracted elements are **hallucinations**.

## Data

Process steps where extracted has the right elements but in a **different order**,
plus realistic errors (wrong value, missing element, extra element).

In [1]:
GOLD = [
    # Record 0: one step, perfect match expected
    {"steps": [
        {"name": "deposit", "temp": 600},
    ]},
    # Record 1: two steps, extracted swaps order + wrong temp + extra step
    {"steps": [
        {"name": "deposit", "temp": 300},
        {"name": "anneal", "temp": 500},
    ]},
    # Record 2: three steps, extracted swaps order + missing step
    {"steps": [
        {"name": "deposit", "temp": 300},
        {"name": "anneal", "temp": 500},
        {"name": "clean", "temp": 100},
    ]},
]

EXTRACTED = [
    {"steps": [
        {"name": "deposit", "temp": 600},
    ]},
    {"steps": [
        {"name": "anneal", "temp": 480},    # swapped + wrong temp
        {"name": "deposit", "temp": 300},   # swapped but correct
        {"name": "cool", "temp": 50},       # hallucinated step
    ]},
    {"steps": [
        {"name": "anneal", "temp": 500},    # swapped but correct
        {"name": "deposit", "temp": 300},   # swapped but correct
        # "clean" is missing -> omission
    ]},
]

## Helper: display results

In [2]:
from struct_extract_eval import evaluate
from example_utils import show_run


## Strategy 1: Ordered (default)

Pairs by position: No `x-eval-align` needed -- this is the default when the key is absent.

e.g. time series, ordered instructions.

In [3]:
ORDERED_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            # No x-eval-align -> ordered (positional) matching
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_ordered = evaluate(GOLD, EXTRACTED, schema=ORDERED_SCHEMA)
show_run(run_ordered, "Ordered")

Ordered
  mean P=0.33  R=0.33  F1=0.33   (3 record(s))
  record  path            gold       extracted  score  status         reason
  0       steps[0].name   'deposit'  'deposit'  1.0    match
  0       steps[0].temp   600        600        1.0    match
  1       steps[0].name   'deposit'  'anneal'   0.0    mismatch       mismatch
  1       steps[0].temp   300        480        0.0    mismatch       values differ without tolerance
  1       steps[1].name   'anneal'   'deposit'  0.0    mismatch       mismatch
  1       steps[1].temp   500        300        0.0    mismatch       values differ without tolerance
  1       steps[-1].name  None       'cool'     0.0    hallucination
  1       steps[-1].temp  None       50         0.0    hallucination
  2       steps[0].name   'deposit'  'anneal'   0.0    mismatch       mismatch
  2       steps[0].temp   300        500        0.0    mismatch       values differ without tolerance
  2       steps[1].name   'anneal'   'deposit'  0.0    mismatch  

Records 1 and 2 score poorly because the extractor swapped the element order.

**Note**: steps[-1] means this element in the array of extracted data is hallucination.

## Strategy 2: Key-Field Matching

Match elements by the value of a unique identifier field. Add `x-eval-align` to the
array node:

```json
"x-eval-align": {"match_by": "key_field", "key": "name"}
```

Gold's `"deposit"` pairs with extracted's `"deposit"`, regardless of position.

Use when elements have a natural unique key (name, id, formula).

In [4]:
KEYFIELD_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "x-eval-align": {"match_by": "key_field", "key": "name"},
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_keyfield = evaluate(GOLD, EXTRACTED, schema=KEYFIELD_SCHEMA)
show_run(run_keyfield, "Key-field")

Key-field
  mean P=0.83  R=0.81  F1=0.80   (3 record(s))
  record  path            gold       extracted  score  status         reason
  0       steps[0].name   'deposit'  'deposit'  1.0    match
  0       steps[0].temp   600        600        1.0    match
  1       steps[0].name   'deposit'  'deposit'  1.0    match
  1       steps[0].temp   300        300        1.0    match
  1       steps[1].name   'anneal'   'anneal'   1.0    match
  1       steps[1].temp   500        480        0.0    mismatch       values differ without tolerance
  1       steps[-1].name  None       'cool'     0.0    hallucination
  1       steps[-1].temp  None       50         0.0    hallucination
  2       steps[0].name   'deposit'  'deposit'  1.0    match
  2       steps[0].temp   300        300        1.0    match
  2       steps[1].name   'anneal'   'anneal'   1.0    match
  2       steps[1].temp   500        500        1.0    match
  2       steps[2].name   'clean'    None       0.0    omission
  2       ste

Score improved. The swapped elements now match correctly by name:
- Record 1: `deposit` pairs with `deposit` (match), `anneal` pairs with `anneal` (name match, temp mismatch). `cool` has no gold counterpart -- hallucination.
- Record 2: `deposit` and `anneal` both match. `clean` has no extracted counterpart -- omission.

## Strategy 3: Hungarian Matching

When elements don't have a unique key field (e.g. arrays of primitives, or objects
without a natural ID), use Hungarian bipartite matching. It finds the optimal pairing
that maximizes total F1 across all pairs.

```json
"x-eval-align": {"match_by": "hungarian"}
```

Requires `scipy` (`pip install scipy`).

In [5]:
HUNGARIAN_SCHEMA = {
    "type": "object",
    "properties": {
        "steps": {
            "type": "array",
            "x-eval-align": {"match_by": "hungarian"},
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "x-eval-compare": "exact"},
                    "temp": {"type": "number", "x-eval-compare": "numeric"},
                },
            },
        },
    },
}

run_hungarian = evaluate(GOLD, EXTRACTED, schema=HUNGARIAN_SCHEMA)
show_run(run_hungarian, "Hungarian")

Hungarian
  mean P=0.83  R=0.81  F1=0.80   (3 record(s))
  record  path            gold       extracted  score  status         reason
  0       steps[0].name   'deposit'  'deposit'  1.0    match
  0       steps[0].temp   600        600        1.0    match
  1       steps[0].name   'deposit'  'deposit'  1.0    match
  1       steps[0].temp   300        300        1.0    match
  1       steps[1].name   'anneal'   'anneal'   1.0    match
  1       steps[1].temp   500        480        0.0    mismatch       values differ without tolerance
  1       steps[-1].name  None       'cool'     0.0    hallucination
  1       steps[-1].temp  None       50         0.0    hallucination
  2       steps[0].name   'deposit'  'deposit'  1.0    match
  2       steps[0].temp   300        300        1.0    match
  2       steps[1].name   'anneal'   'anneal'   1.0    match
  2       steps[1].temp   500        500        1.0    match
  2       steps[2].name   'clean'    None       0.0    omission
  2       ste

Hungarian produces the same result as key-field here because the `name` field provides
enough signal for optimal matching. The difference matters when there's no obvious key.

## Other Example: Hungarian with Primitive Arrays

Hungarian shines with arrays of primitives where key-field matching isn't possible.

In [6]:
TAGS_GOLD = [{"tags": ["silicon", "thin-film", "CVD"]}]
TAGS_EXTRACTED = [{"tags": ["CVD", "silicon", "PVD"]}]

TAGS_ORDERED = {
    "type": "object",
    "properties": {
        "tags": {
            "type": "array",
            "items": {"type": "string", "x-eval-compare": "exact"},
        },
    },
}

TAGS_HUNGARIAN = {
    "type": "object",
    "properties": {
        "tags": {
            "type": "array",
            "x-eval-align": {"match_by": "hungarian"},
            "items": {"type": "string", "x-eval-compare": "exact"},
        },
    },
}

run_tags_ord = evaluate(TAGS_GOLD, TAGS_EXTRACTED, schema=TAGS_ORDERED)
run_tags_hun = evaluate(TAGS_GOLD, TAGS_EXTRACTED, schema=TAGS_HUNGARIAN)

print(f"Ordered:   F1={run_tags_ord.mean_f1:.3f}  (positional: 'silicon' vs 'CVD' = mismatch)")
print(f"Hungarian: F1={run_tags_hun.mean_f1:.3f}  ('silicon' matches 'silicon', 'CVD' matches 'CVD')")
print()
show_run(run_tags_hun)

Ordered:   F1=0.000  (positional: 'silicon' vs 'CVD' = mismatch)
Hungarian: F1=0.667  ('silicon' matches 'silicon', 'CVD' matches 'CVD')

  mean P=0.67  R=0.67  F1=0.67   (1 record(s))
  record  path      gold         extracted  score  status         reason
  0       tags[0]   'silicon'    'silicon'  1.0    match
  0       tags[2]   'CVD'        'CVD'      1.0    match
  0       tags[1]   'thin-film'  None       0.0    omission
  0       tags[-1]  None         'PVD'      0.0    hallucination
